# Day 16: Student Wellbeing Statistical Analysis and Probability

This notebook analyzes `Day16_Student_Wellbeing_Survey.csv` using descriptive statistics, the IQR outlier method, probability, independence, Bayes' theorem, and the normal distribution. Variance and standard deviation use the sample convention (`ddof=1`).

In [1]:
import pandas as pd
import numpy as np
from IPython.display import display

file_path = 'Day16_Student_Wellbeing_Survey.csv'
df = pd.read_csv(file_path)
print(f'Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns')
display(df.head())
print('Missing values:', int(df.isna().sum().sum()))
print('Year of study counts:')
display(df['Year_of_Study'].value_counts().sort_index().rename('count'))

Dataset shape: 600 rows x 20 columns


,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
0,STU0001,21,Data Science,3,Bengaluru,Hostel,No,Yes,Good,Café,15.2,5.7,3.2,3,11.8,5.4,63.1,3.2,5131.0,7.8
1,STU0002,21,Data Science,4,Pune,Hostel,Yes,No,Poor,Library,17.6,7.7,3.5,4,20.3,2.1,74.2,3.8,3508.0,9.2
2,STU0003,22,Life Sciences,2,Chandigarh,Home,Yes,No,Poor,Home,18.9,6.4,2.8,1,15.0,5.2,77.3,4.0,4584.0,7.7
3,STU0004,18,Business,4,Bengaluru,Hostel,Yes,Yes,Average,Café,19.5,6.7,3.0,5,12.1,4.7,67.0,4.0,5456.0,5.3
4,STU0005,20,Business,3,Jammu,Shared Apartment,Yes,No,Good,Home,18.4,7.3,3.0,1,30.2,5.4,65.6,3.2,11732.0,15.9


Missing values: 0
Year of study counts:


Year_of_Study
1    164
2    159
3    157
4    120
Name: count, dtype: int64

## 1. Descriptive statistics

Formulae: range = max - min; sample variance $s^2 = \frac{\sum(x_i-\bar{x})^2}{n-1}$; sample standard deviation $s = \sqrt{s^2}$; IQR = Q3 - Q1. The mode is the most frequent observed value.

In [2]:
descriptive_columns = [
    'Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours',
    'Stress_Score', 'Academic_Readiness_Score'
]
summary = pd.DataFrame(index=descriptive_columns)
summary['mean'] = df[descriptive_columns].mean()
summary['median'] = df[descriptive_columns].median()
summary['mode'] = [df[column].mode().iloc[0] for column in descriptive_columns]
summary['range'] = df[descriptive_columns].max() - df[descriptive_columns].min()
summary['variance'] = df[descriptive_columns].var(ddof=1)
summary['standard_deviation'] = df[descriptive_columns].std(ddof=1)
summary['Q1'] = df[descriptive_columns].quantile(0.25)
summary['Q3'] = df[descriptive_columns].quantile(0.75)
summary['IQR'] = summary['Q3'] - summary['Q1']
print('Formula implemented: range = max - min; variance = sample variance; IQR = Q3 - Q1')
display(summary.round(3))
greatest_std = summary['standard_deviation'].idxmax()
greatest_variance = summary['variance'].idxmax()
std_value = summary.loc[greatest_std, 'standard_deviation']
variance_value = summary.loc[greatest_variance, 'variance']
print(f'Greatest variability by standard deviation: {greatest_std} ({std_value:.3f})')
print(f'Greatest variability by raw variance: {greatest_variance} ({variance_value:.3f})')
print('Interpretation: Academic readiness has the widest spread among the five variables by native-unit standard deviation; raw variance should only be compared cautiously because units differ.')

Formula implemented: range = max - min; variance = sample variance; IQR = Q3 - Q1


,mean,median,mode,range,variance,standard_deviation,Q1,Q3,IQR
Weekly_Study_Hours,15.691,15.40,13.1,30.0,19.217,4.384,13.000,18.425,5.425
Average_Sleep_Hours,6.998,7.00,7.0,5.0,0.703,0.839,6.475,7.600,1.125
Daily_Screen_Time_Hours,4.503,4.20,3.5,11.2,3.469,1.863,3.200,5.400,2.200
Stress_Score,4.464,4.50,4.7,8.4,2.782,1.668,3.300,5.700,2.400
Academic_Readiness_Score,71.773,71.65,71.1,56.1,93.982,9.694,65.200,78.025,12.825


Greatest variability by standard deviation: Academic_Readiness_Score (9.694)
Greatest variability by raw variance: Academic_Readiness_Score (93.982)
Interpretation: Academic readiness has the widest spread among the five variables by native-unit standard deviation; raw variance should only be compared cautiously because units differ.


## 2. IQR outlier detection

For each variable, lower fence = Q1 - 1.5 x IQR and upper fence = Q3 + 1.5 x IQR. Values below the lower fence or above the upper fence are flagged as outliers.

In [3]:
outlier_columns = [
    'Weekly_Study_Hours', 'Daily_Screen_Time_Hours',
    'Commute_Time_Minutes', 'Monthly_Discretionary_Spending'
]
outlier_rows = []
outlier_masks = {}
for column in outlier_columns:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[column] < lower) | (df[column] > upper)
    outlier_masks[column] = mask
    outlier_rows.append({'variable': column, 'Q1': q1, 'Q3': q3, 'IQR': iqr, 'lower_fence': lower, 'upper_fence': upper, 'outlier_count': int(mask.sum()), 'outlier_values': df.loc[mask, column].tolist()})
outlier_summary = pd.DataFrame(outlier_rows).set_index('variable')
display(outlier_summary.round(3))

study_mask = outlier_masks['Weekly_Study_Hours']
study_without_outliers = df.loc[~study_mask, 'Weekly_Study_Hours']
comparison = pd.DataFrame({
    'mean': [df['Weekly_Study_Hours'].mean(), study_without_outliers.mean()],
    'median': [df['Weekly_Study_Hours'].median(), study_without_outliers.median()],
}, index=['before removing outliers', 'after removing outliers'])
display(comparison.round(3))
print(f'Weekly study hours: {int(study_mask.sum())} outlier(s) detected.')
print('Interpretation: the before/after table shows how the detected extreme study-hour values affect the center of the distribution.')

,Q1,Q3,IQR,lower_fence,upper_fence,outlier_count,outlier_values
variable,,,,,,,
Weekly_Study_Hours,13.000,18.425,5.425,4.863,26.562,8,"[27.8, 26.6, 26.6, 4.0, 27.2, 34.0, 4.3, 27.2]"
Daily_Screen_Time_Hours,3.200,5.400,2.200,-0.100,8.700,18,"[9.8, 10.1, 11.8, 10.4, 12.4, 12.0, 10.6, 11.3..."
Commute_Time_Minutes,13.375,31.525,18.150,-13.850,58.750,4,"[65.2, 62.3, 72.5, 118.0]"
Monthly_Discretionary_Spending,4113.000,7808.500,3695.500,-1430.250,13351.750,20,"[28000.0, 16791.0, 13945.0, 14064.0, 31500.0, ..."


,mean,median
before removing outliers,15.691,15.40
after removing outliers,15.603,15.35


Weekly study hours: 8 outlier(s) detected.
Interpretation: the before/after table shows how the detected extreme study-hour values affect the center of the distribution.


## 3. Probability events

Event A: Part_Time_Job = Yes. Event B: Stress_Score >= 7. Event C: Scholarship = Yes. Event D: Exercise_Days_Per_Week >= 3.

Formulae: $P(E)=n(E)/n$; $P(A\text{ or }B)=P(A\cup B)$; $P(A\text{ and }B)=P(A\cap B)$; $P(A|B)=P(A\cap B)/P(B)$.

In [4]:
event_a = df['Part_Time_Job'].eq('Yes')
event_b = df['Stress_Score'].ge(7)
event_c = df['Scholarship'].eq('Yes')
event_d = df['Exercise_Days_Per_Week'].ge(3)
n = len(df)
p_a = event_a.mean()
p_b = event_b.mean()
p_c = event_c.mean()
p_d = event_d.mean()
p_a_or_b = (event_a | event_b).mean()
p_a_and_b = (event_a & event_b).mean()
p_a_given_b = p_a_and_b / p_b
p_b_given_a = p_a_and_b / p_a
probabilities = pd.Series({
    'P(A)': p_a, 'P(B)': p_b, 'P(C)': p_c, 'P(D)': p_d,
    'P(A or B)': p_a_or_b, 'P(A and B)': p_a_and_b,
    'P(A|B)': p_a_given_b, 'P(B|A)': p_b_given_a
})
display(probabilities.to_frame('probability').round(4))

year_1 = df['Year_of_Study'].eq(1)
year_4 = df['Year_of_Study'].eq(4)
print(f'Year 1 and Year 4 intersection count: {(year_1 & year_4).sum()}')
print('Year_of_Study = 1 and Year_of_Study = 4 are mutually exclusive because no student can belong to both categories.')
product = p_a * p_b
print(f'Independence check: P(A and B) = {p_a_and_b:.4f}; P(A) x P(B) = {product:.4f}; difference = {p_a_and_b - product:.4f}')
print('Interpretation:', 'The events appear independent in this sample because the two values are close.' if np.isclose(p_a_and_b, product, atol=0.01) else 'The events do not appear independent because the two values differ materially.')

,probability
P(A),0.2533
P(B),0.0750
P(C),0.3067
P(D),0.5933
P(A or B),0.2767
P(A and B),0.0517
P(A|B),0.6889
P(B|A),0.2039


Year 1 and Year 4 intersection count: 0
Year_of_Study = 1 and Year_of_Study = 4 are mutually exclusive because no student can belong to both categories.
Independence check: P(A and B) = 0.0517; P(A) x P(B) = 0.0190; difference = 0.0327
Interpretation: The events do not appear independent because the two values differ materially.


## 4. Bayes' theorem

Bayes' theorem: $P(A|B)=\frac{P(B|A)P(A)}{P(B|A)P(A)+P(B|\neg A)P(\neg A)}$. The result is checked against the direct calculation above.

In [5]:
p_not_a = 1 - p_a
p_b_given_not_a = (event_b & ~event_a).sum() / (~event_a).sum()
p_a_given_b_bayes = (p_b_given_a * p_a) / ((p_b_given_a * p_a) + (p_b_given_not_a * p_not_a))
bayes_values = pd.Series({
    'P(B|A)': p_b_given_a, 'P(B|not A)': p_b_given_not_a,
    'P(A)': p_a, 'P(not A)': p_not_a,
    'P(A|B) by Bayes': p_a_given_b_bayes,
    'P(A|B) direct': p_a_given_b
})
display(bayes_values.to_frame('value').round(6))
print(f'Agreement: {np.isclose(p_a_given_b_bayes, p_a_given_b)}')
print('Interpretation: Bayes theorem agrees with the direct conditional probability, confirming the calculation.')

,value
P(B|A),0.203947
P(B|not A),0.031250
P(A),0.253333
P(not A),0.746667
P(A|B) by Bayes,0.688889
P(A|B) direct,0.688889


Agreement: True
Interpretation: Bayes theorem agrees with the direct conditional probability, confirming the calculation.


## 5. Normal distribution analysis

For Academic_Readiness_Score, $z=(x-\mu)/\sigma$. The empirical rule expects about 68%, 95%, and 99.7% of observations within 1, 2, and 3 standard deviations of the mean, respectively.

In [6]:
readiness = df['Academic_Readiness_Score']
mu = readiness.mean()
sigma = readiness.std(ddof=1)
highest_index = readiness.idxmax()
lowest_index = readiness.idxmin()
highest_score = readiness.loc[highest_index]
lowest_score = readiness.loc[lowest_index]
highest_z = (highest_score - mu) / sigma
lowest_z = (lowest_score - mu) / sigma
print(f'Mean = {mu:.3f}; sample standard deviation = {sigma:.3f}')
highest_student_id = df.loc[highest_index, 'Student_ID']
lowest_student_id = df.loc[lowest_index, 'Student_ID']
print(f'Highest score: {highest_score:.1f} ({highest_student_id}), Z = {highest_z:.3f}')
print(f'Lowest score: {lowest_score:.1f} ({lowest_student_id}), Z = {lowest_z:.3f}')
print('Interpretation: the highest-scoring student is about {:.2f} standard deviations above the mean; the lowest is about {:.2f} standard deviations below the mean.'.format(highest_z, abs(lowest_z)))

empirical = pd.DataFrame({
    'within_standard_deviations': [1, 2, 3],
    'expected_percentage': [68.0, 95.0, 99.7],
    'observed_percentage': [((readiness >= mu - k*sigma) & (readiness <= mu + k*sigma)).mean() * 100 for k in [1, 2, 3]]
})
display(empirical.round(2))
print('Interpretation: the expected percentages are theoretical normal-distribution benchmarks; observed percentages show how closely this sample follows them.')

Mean = 71.773; sample standard deviation = 9.694
Highest score: 98.0 (STU0131), Z = 2.705
Lowest score: 41.9 (STU0543), Z = -3.081
Interpretation: the highest-scoring student is about 2.71 standard deviations above the mean; the lowest is about 3.08 standard deviations below the mean.


,within_standard_deviations,expected_percentage,observed_percentage
0,1,68.0,69.50
1,2,95.0,95.17
2,3,99.7,99.83


Interpretation: the expected percentages are theoretical normal-distribution benchmarks; observed percentages show how closely this sample follows them.


## 6. Five meaningful statistical observations

1. Academic readiness is the most variable of the five requested measures by sample standard deviation ($s=9.694$), while sleep is the most consistent ($s=0.839$).
2. The IQR method flags 8 Weekly_Study_Hours, 18 Daily_Screen_Time_Hours, 4 Commute_Time_Minutes, and 20 Monthly_Discretionary_Spending observations. Removing the 8 study-hour outliers changes the study-hour mean from 15.691 to 15.603 and the median from 15.400 to 15.350.
3. Part-time work occurs for 25.33% of students, high stress for 7.50%, scholarships for 30.67%, and exercising at least three days weekly for 59.33%.
4. Part-time work and high stress do not appear independent in this sample: $P(A\\cap B)=0.0517$ is much larger than $P(A)P(B)=0.0190$. Among high-stress students, 68.89% have a part-time job, compared with 25.33% overall.
5. Readiness scores are close to the empirical-rule pattern: 69.50%, 95.17%, and 99.83% fall within 1, 2, and 3 standard deviations. The maximum score is 2.705 SD above the mean, while the minimum is 3.081 SD below it, showing a slightly more extreme low tail.